Using the Carlifonia Dataset to tackle a Regression Neural Network

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()

X_train_full, X_test, y_train_full , y_test = train_test_split(housing.data, housing.target, random_state=42)

X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)



In [2]:
# a) Using the Sequential API
# The output layer has a s single neuron since we are only predicting a single value
# We use no activation function
# Loss function is the mean squared error

from tensorflow import keras

model = keras.models.Sequential([
    keras.layers.Dense(30, activation = 'relu', input_shape= X_train.shape[1:]),
    keras.layers.Dense(1)
])

model.compile(loss='mean_squared_error', optimizer='sgd')
history =  model.fit(X_train, y_train, epochs=20, validation_data=(X_valid,y_valid))

mse_test = model.evaluate(X_test, y_test)
X_new = X_test[:3]
y_pred = model.predict(X_new)


2026-01-29 14:11:52.147932: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-29 14:11:53.726763: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-29 14:12:03.487981: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/daniel/Documents/DEEP LEARNING/deep_learning/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-01-29 14:1

Epoch 1/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.8281 - val_loss: 1.5469
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.8858 - val_loss: 0.5131
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.4672 - val_loss: 0.4730
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.4451 - val_loss: 0.4454
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.4249 - val_loss: 0.4321
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.4167 - val_loss: 0.4203
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.4182 - val_loss: 0.4188
Epoch 8/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.3994 - val_loss: 0.4113
Epoch 9/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.3958 - val_loss: 0.4046
Epoch 10/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.3895 - val_loss: 0.4038
Epoch 11/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.3862 - val_loss: 0.3970
Epoch 12/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 

In [3]:
y_pred

array([[0.6954154],
       [1.5596396],
       [3.725582 ]], dtype=float32)

In [8]:
# b) Using the Functional API, a complex model 
# It connects all parts of an input directly to an output layer
# This architecture makes it possible for neural network to learn both deep patterns (using the deep path) and simple rules (using the short path) 
# In contrast a regular MLP forces all the data to flow through the full stack of layers thus simple  patterns in rhe data may end up distorted by this sequance of transformations.add()


input_ = keras.layers.Input(shape=X_train.shape[1:])
hidden1 = keras.layers.Dense(30, activation='relu')(input_)
hidden2 = keras.layers.Dense(30, activation='relu')(hidden1)
concat = keras.layers.Concatenate()([input_, hidden2])
output = keras.layers.Dense(1)(concat)

model = keras.models.Model(inputs=[input_], outputs=[output])

In [9]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 30)        │        270 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 30)        │        930 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 38)        │          0 │ input_layer_3[0]… │
│ (Concatenate)       │                   │            │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         39 │ concatenate_2[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,239 (4.84 KB)

 Trainable params: 1,239 (4.84 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(loss='mean_squared_error', optimizer=keras.optimizers.SGD(learning_rate=1e-3))
history = model.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid))

mse_test = model.evaluate(X_test, y_test)
y_pred = model.predict(X_new)

Epoch 1/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.8388 - val_loss: 1.3229
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.7781 - val_loss: 0.7126
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6773 - val_loss: 0.6533
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6253 - val_loss: 0.6117
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5880 - val_loss: 0.5788
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5584 - val_loss: 0.5592
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5381 - val_loss: 0.5390
Epoch 8/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5199 - val_loss: 0.5245
Epoch 9/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5063 - val_loss: 0.5093
Epoch 10/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4948 - val_loss: 0.5093
Epoch 11/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4840 - val_loss: 0.4894
Epoch 12/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

In [ ]:
# What if I want to send different subsets of input features through the wide or deep paths? I will send 5 features (features 0-4 ) and 6 through the deep path (features 2 to 7). 
# Note that 3 features will go through both (features 2,3 and 4)

import numpy as np
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)


input_A =  keras.layers.Input(shape=[5], name='wide_input')
input_B = keras.layers.Input(shape=[6], name='deep_input')
hidden1 = keras.layers.Dense(30, activation='relu')(input_B)
hidden2 = keras.layers.Dense(30, activation='relu')(hidden1)
concat = keras.layers.concatenate([input_A, hidden2])
output = keras.layers.Dense(1, name='output')(concat)

model = keras.models.Model(inputs=[input_A, input_B], outputs=[output])

In [13]:
model.compile(loss="mse", optimizer=keras.optimizers.SGD(learning_rate=1e-3))

X_train_A, X_train_B = X_train[:, :5], X_train[:, 2:]
X_valid_A, X_valid_B = X_valid[:, :5], X_valid[:, 2:]
X_test_A, X_test_B = X_test[:, :5], X_test[:, 2:]
X_new_A, X_new_B = X_test_A[:3], X_test_B[:3]

history =  model.fit((X_train_A, X_train_B), y_train, epochs=20, validation_data=((X_valid_A, X_valid_B), y_valid))

mse_test = model.evaluate((X_test_A, X_test_B), y_test)
y_pred = model.predict((X_new_A, X_new_B))

Epoch 1/20


363/363 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.8191 - val_loss: 0.7899
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.6820 - val_loss: 0.6781
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.6216 - val_loss: 0.6208
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.5849 - val_loss: 0.5860
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.5592 - val_loss: 0.5626
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.5396 - val_loss: 0.5459
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.5241 - val_loss: 0.5334
Epoch 8/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.5118 - val_loss: 0.5236
Epoch 9/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.5018 - val_loss: 0.5159
Epoch 10/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.4935 - val_loss: 0.5097
Epoch 11/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.4863 - val_loss: 0.5041
Epoch 12/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step

In [14]:
# Adding an auxiliary output for regularization



input_A = keras.layers.Input(shape=[5], name="wide_input")
input_B = keras.layers.Input(shape=[6], name="deep_input")
hidden1 = keras.layers.Dense(30, activation="relu")(input_B)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.concatenate([input_A, hidden2])
output = keras.layers.Dense(1, name="main_output")(concat)
aux_output = keras.layers.Dense(1, name="aux_output")(hidden2)



In [15]:

model = keras.models.Model(inputs=[input_A, input_B], outputs=[output, aux_output])

In [16]:

model.compile(loss=['mse', 'mse'], loss_weights=[0.9, 0.1] , optimizer=keras.optimizers.SGD(learning_rate=1e-3))

In [17]:
history = model.fit([X_train_A, X_train_B], [y_train, y_train], epochs=20, validation_data=([X_valid_A, X_valid_B], [y_valid, y_valid]))

Epoch 1/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - aux_output_loss: 3.5904 - loss: 2.4695 - main_output_loss: 2.3444 - val_aux_output_loss: 2.7476 - val_loss: 1.2440 - val_main_output_loss: 1.0768
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - aux_output_loss: 2.2031 - loss: 1.0200 - main_output_loss: 0.8884 - val_aux_output_loss: 2.1105 - val_loss: 0.9198 - val_main_output_loss: 0.7874
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - aux_output_loss: 1.7879 - loss: 0.8530 - main_output_loss: 0.7490 - val_aux_output_loss: 1.8259 - val_loss: 0.8249 - val_main_output_loss: 0.7136
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - aux_output_loss: 1.5937 - loss: 0.7869 - main_output_loss: 0.6972 - val_aux_output_loss: 1.6695 - val_loss: 0.7776 - val_main_output_loss: 0.6784
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - aux_output_loss: 1.4869 - loss: 0.7474 - main_output_loss: 0.6652 - val_aux_output_loss: 1.5668 - val_loss: 0.7450 - val_main_output_loss: 0.

In [18]:
# Evaluating the model

total_loss, main_loss, aux_loss = model.evaluate(
    [X_test_A, X_test_B], [y_test, y_test]
    )

162/162 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - aux_output_loss: 1.0027 - loss: 0.5646 - main_output_loss: 0.5169


In [19]:
y_pred_main, y_pred_aux = model.predict([X_new_A, X_new_B])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step


In [ ]:
# USING THE SUBCLASS API TO BUILD DYNAMIC MODELS
# This is mainly for models involving loops, varying shapes , conditional branching and other dynamic behaviour.
# It uses a class

class WideAndDeepModel(keras.Model):
    def __init__(self, units=30, activation="relu", **kwargs):
        super().__init__(**kwargs)  # Handles standard args eg name
        self.hidden1 = keras.layers.Dense(units, activation=activation)
        self.hidden2 = keras.layers.Dense(units, activation=activation)
        self.main_output = keras.layers.Dense(1)
        self.aux_output = keras.layers.Dense(1)
        
        
    def call(self, inputs):
        input_A, input_B = inputs
        hidden1 = self.hidden1(input_B)
        hidden2 = self.hidden2(hidden1)
        concat = keras.layers.Concaenate([input_A, hidden2])
        main_output = self.main_output(concat)
        aux_output = self.aux_output(hidden2)
        return main_output, aux_output
    
model = WideAndDeepModel()